# 第 5 章｜Tool + Resource = AI Agent

依序執行每一格；可修改標示的參數後重跑。

## 執行前：可替換設定總覽

以下項目都可以依測試環境替換：

- `.env` Provider：正式課程填 `CILLM_API_KEY`；只有個人本機測試才填 `OPENAI_API_KEY`；兩者都有時 CILLM 優先。
- CILLM：`CILLM_BASE_URL`、`CILLM_USER_ID`、`CILLM_PLATFORM`、`CILLM_AGENT`、`GPT_OSS_MODEL_NAME`、`GEMMA_MODEL_NAME`、`CILLM_VISION_FORMAT`。
- CILLM 圖片：目前保留 `CILLM_VISION_FORMAT=html` 相容模式，不要求重新部署 CILLM gateway。
- OpenAI：`OPENAI_API_KEY`、`OPENAI_MODEL_NAME`；預設 `gpt-4o`，僅供個人筆電模擬測試。
- 密碼式 AES：第 6 章由使用者在 Notebook 隱藏輸入設定保險庫密碼；密碼不寫入 `.env` 或加密檔。
- 路徑：只有從其他工作目錄啟動 Notebook 時才需要調整 `ROOT`；一般從教材根目錄或 `notebooks/` 啟動不必修改。

> 請勿把含有真實 Key 的 `.env`、Notebook 輸出或截圖提交到 Git。

### 本章可替換

- `q1`、`q2`、`USER_REQUEST`：替換 Tool、Resource 與整合 Agent 的問題。
- `threshold`：替換航班延誤篩選門檻。
- `resource_name`：替換要查詢的 Resource。
- Excel 路徑與 pandas `code`：替換資料檔、欄位、條件與輸出欄位；程式必須設定 `result`。
- `ask_gpt_oss` 的 instruction：替換最終回答角色、格式與限制。

In [24]:
from pathlib import Path
import importlib, os, sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import course_utils
importlib.reload(course_utils)
from course_utils import *
print("教材根目錄：", ROOT)
connection = verify_cillm_key()
print("✅ API Key 驗證成功")
print("目前 Provider：", connection["provider"])
print("目前模型：", connection["model"])
print("首次回覆：", connection["reply"])

教材根目錄： C:\Users\rathe\Project\cillm\CILLM_Workshop\Lecture03
✅ API Key 驗證成功
目前 Provider： openai
目前模型： gpt-4o
首次回覆： CILLM 連線成功


## 檢查目前 API Key 的基本 Scope

下方 Cell 會查詢並列出這支 Key 實際具備的 RBAC scopes。本教材呼叫 GPT-OSS 至少需要 `llm.chat`。

In [25]:
scopes = get_current_key_scopes()
if scopes is None:
    print("目前使用 OpenAI API；CILLM RBAC scope 不適用。")
else:
    print("目前 CILLM_API_KEY 具備的 scopes：")
    for scope in scopes:
        print("-", scope)
    required_scope = "llm.chat"
    print(f"✅ 已具備教材基本 scope：{required_scope}" if "*" in scopes or required_scope in scopes else f"❌ 缺少教材基本 scope：{required_scope}")

目前使用 OpenAI API；CILLM RBAC scope 不適用。


> **執行模式 Hint**
>
> - 本教材每章第一格會取得 `CILLM_API_KEY`；若 `.env` 未設定，Notebook 會以隱藏輸入提示使用者填入。
> - 取得 Key 後會立即呼叫一次 `openai/gpt-oss-120b`，確認 Key 與連線可用。
> - `CILLM_BASE_URL` 預設沿用 Lecture 02 的測試端點；缺少或無效 Key 時會立即停止。
> - 正式課程的 CILLM 模式：文字使用 GPT-OSS，圖片使用 NVIDIA NIM 的 `google/gemma-4-31b-it`，不會 fallback 到 OpenAI。
> - `CILLM_VISION_FORMAT=html` 保留現行 gateway 的字串 content 相容性，不需要重新部署 GPU server。
> - 本教材暫不啟用需要多模態陣列 schema 的 `blocks` 模式。
> - 只有 OpenAI Key 時，才以 `gpt-4o` 模擬文字與圖片流程，供個人筆電測試。

> **Agent Hint**
>
> - 問純計算：預期只用 Tool。
> - 問公司規範：預期只讀 Resource。
> - 問「哪些 Excel 航班符合補償，以及提供什麼服務」：預期先讀規範，再執行 Excel Tool。
> - 問圖片中的班號是否符合延誤規範：可觀察圖片 Tool 加 Resource 的多步驟需求。

## 每個案例都先用 AgentRoutePlan 判斷 Tool / Resource

案例問題雖然固定，但 Tool 與 Resource 選擇不寫死。

In [26]:
# 案例 1：預期只用 Tool，但仍由 AI + Pydantic 判斷
q1 = "312 個座位、載客率 87%，精確計算旅客數。"
plan1 = route_agent(q1)
show(plan1.model_dump())
tool_route1 = route_tool(q1)
args1 = validate_tool_arguments(tool_route1)
if tool_route1.tool_name != "math_tool": raise RuntimeError(f"AI 未選擇 math_tool：{tool_route1.model_dump()}")
r1 = math_tool(args1.a, args1.op, args1.b)
print_execution_trace(question=q1, tool=f"✓ {tool_route1.tool_name} (Pydantic routed)", resource=plan1.resources, tool_result=r1, answer=f"計算結果：{r1}")

{
  "tools": [
    "math_tool"
  ],
  "resources": [],
  "reason": "問題需要精確計算載客率下的旅客數，適合使用 math_tool 進行數值運算。"
}
AI Agent 執行追蹤
使用者問題：
312 個座位、載客率 87%，精確計算旅客數。

是否需要工具：
✓ math_tool (Pydantic routed)

是否需要資源：
[]

工具執行結果：
271.44

最終回答：
計算結果：271.44



In [27]:
# 案例 2：預期只用 Resource，但仍由 AI + Pydantic 判斷
q2 = "特殊餐點要多久前申請？"
plan2 = route_agent(q2)
show(plan2.model_dump())
resource_route2 = route_resource(q2)
if resource_route2.resource_name == "none": raise RuntimeError("AI 未選擇 Resource")
rn2 = resource_route2.resource_name
ctx2 = search_resource(rn2, q2)
answer2 = ask_gpt_oss(q2, ctx2, "只能根據 Resource 回答。")
print_execution_trace(question=q2, tool=plan2.tools, resource=f"✓ {rn2} (Pydantic routed)", answer=answer2)

{
  "tools": [],
  "resources": [
    "passenger_service_rules"
  ],
  "reason": "特殊餐點的申請時間屬於旅客服務規定的一部分，因此需要查閱 'passenger_service_rules' 來獲取準確的資訊。"
}
AI Agent 執行追蹤
使用者問題：
特殊餐點要多久前申請？

是否需要工具：
[]

是否需要資源：
✓ passenger_service_rules (Pydantic routed)

最終回答：
特殊餐點須於起飛前 24 小時申請。



In [28]:
# 案例 3：Tool + Resource 都由 AI + Pydantic 判斷
# 使用者只需提問；Agent 會自動盤點可用的檔案、Excel 欄位與 Resources。
USER_REQUEST = "哪些航班符合延誤補償條件，以及應該提供什麼服務？"

print("Agent 自動發現的可用環境：\n")
print(describe_available_context())
print("\n" + "=" * 70)

plan3 = route_agent(USER_REQUEST)
print("AgentRoutePlan："); show(plan3.model_dump())
if "excel_python_tool" not in plan3.tools:
    raise RuntimeError(f"Agent 計畫沒有選 Excel Tool。請檢查問題是否明確要分析 Excel：{plan3.model_dump()}")
if not plan3.resources:
    raise RuntimeError(f"Agent 計畫沒有選 Resource。請檢查問題是否明確要套用規範：{plan3.model_dump()}")

# 步驟 1：先讓 Resource Router 選規範，再取得相關條文
resource_route3 = route_resource(USER_REQUEST)
print("ResourceRouteResult："); show(resource_route3.model_dump())
if resource_route3.resource_name == "none": raise RuntimeError("AI 未選擇 Resource")
resource_name = resource_route3.resource_name
rule = search_resource(resource_name, USER_REQUEST)
print("命中的規範條文：\n", rule)

# 步驟 2：根據 Agent Plan 把複合問題拆成單一 Excel 子任務
# 不再把「規範 + Excel」的原始複合問題丟給 Tool Router 重新判斷。
threshold = 120
TOOL_SUBTASK = f"使用 flight_delays.xlsx 做精確資料處理：篩選 delay_minutes 大於或等於 {threshold} 的航班，回傳 flight 與 delay_minutes。"
TOOL_CONTEXT = f"AgentRoutePlan 已規劃 tools={plan3.tools}。\n相關規範：\n{rule}"
tool_route3 = route_tool(TOOL_SUBTASK, TOOL_CONTEXT)
print("Tool 子任務：", TOOL_SUBTASK)
print("ToolRouteResult："); show(tool_route3.model_dump())
if tool_route3.tool_name != "excel_python_tool":
    raise RuntimeError(f"Excel 子任務未選到 excel_python_tool：{tool_route3.model_dump()}")
validate_tool_arguments(tool_route3)

# 步驟 3：通過 Pydantic 與 RBAC 後才執行 Excel Tool
code = f"df = pd.read_excel(excel_path)\nresult = df.loc[df['delay_minutes'] >= {threshold}, ['flight','delay_minutes']].to_dict('records')"
flights = safe_excel_python(ROOT / "data/excel/flight_delays.xlsx", code)
answer = ask_gpt_oss(USER_REQUEST, "規範：\n"+rule+"\n\nExcel Tool 結果：\n"+json.dumps(flights,ensure_ascii=False), "整合規範與 Tool 結果回答，不可加入未提供的資訊。使用繁體中文。")
print_execution_trace(question=USER_REQUEST, tool=f"✓ {tool_route3.tool_name} (Pydantic routed)", resource=f"✓ {resource_name} (Pydantic routed)", tool_result=flights, answer=answer)

Agent 自動發現的可用環境：

可用 text 檔案：flight_note.txt, maintenance_shift_handover.txt, mixed_language_cabin_log.txt, passenger_feedback.txt, security_incident_report.txt, weather_disruption_notice.txt
可用 images 檔案：fictional_baggage_tag.png, gate_delay_display.png, mock_boarding_pass.png, ramp_safety_inspection.png
可用 audio 檔案：bilingual_boarding_call.wav, maintenance_handover.wav, passenger_service_dialogue.wav, sample_announcement.wav, weather_delay_announcement.wav
可用 Excel：flight_delays.xlsx（delays 欄位=['flight', 'department', 'delay_minutes', 'passengers']）
可用 Resources：company_information: 一般公司資訊、客服時間、樞紐、服務語言與基本公司原則。；passenger_service_rules: 旅客登機、行李、延誤、餐飲券、補償與特殊服務規定。；flight_operations_guide: 航班、航務、起飛、飛航計畫、雷雨與異常通報作業。；maintenance_guidelines: 機務維修、工單、零件、航材、工具與維修安全規定。；it_security_policy: API key、密碼、帳號、資料保護與資訊安全規定。；employee_policy: 員工請假、訓練、設備借用、差旅與加班規定。

AgentRoutePlan：
{
  "tools": [
    "excel_python_tool"
  ],
  "resources": [
    "passenger_service_rules"
  ],
  "reason": "需要從 Excel 表格中篩選出符合延

In [33]:
USER_REQUEST = "哪些航班符合延誤補償條件，以及應該提供什麼服務？"
extension_result = execute_agent(USER_REQUEST)
show(extension_result)

{
  "question": "哪些航班符合延誤補償條件，以及應該提供什麼服務？",
  "plan": {
    "tools": [
      "excel_python_tool"
    ],
    "resources": [
      "passenger_service_rules"
    ],
    "reason": "需要使用 Excel 工具來篩選符合延誤補償條件的航班，並根據公司規定提供相應的服務資訊。"
  },
  "resources": {
    "passenger_service_rules": "4. 延誤達 120 分鐘提供餐飲券與通訊協助。\n5. 延誤達 240 分鐘另協助改班或退票。\n10. 補償資格仍須排除不可抗力與旅客自身因素。\n11. 延誤達 120 分鐘時，餐飲券由機場服務櫃檯依現場作業發放。\n14. 轉機旅客因延誤可能錯失接續航班時，應洽轉機服務櫃檯協助。"
  },
  "generated_codes": {
    "excel_python_tool": {
      "code": "\ndf = pd.read_excel(excel_path)\n\n# 過濾出符合延誤補償條件的航班\ncompensation_eligible_flights = df[df['delay_minutes'] >= 120]\n\n# 定義服務條件\ncompensation_eligible_flights['services'] = compensation_eligible_flights['delay_minutes'].apply(\n    lambda x: '餐飲券與通訊協助' + ('，協助改班或退票' if x >= 240 else '')\n)\n\n# 只選擇航班號和服務\nresult = compensation_eligible_flights[['flight', 'services']]\n",
      "reason": "程式碼首先讀取 Excel 檔案，然後過濾出延誤時間達到 120 分鐘或以上的航班，這些航班符合延誤補償條件。接著，根據延誤時間，為每個符合條件的航班指定應提供的服務：延誤達 120 分鐘提供餐飲券與通訊協助，延誤達 240 分

## 其他 Tool + Resource 題目（預設註解）

下列題目測試 `excel_python_tool` + `flight_operations_guide`。取消註解後，`execute_agent()` 會從規劃一路執行到最終回答，並保留中間結果。

In [32]:
# 題目 B：Excel Tool + 航務 Resource
# 取消下列 3 行註解，會實際執行 Plan → Resource → Generated Code → Excel Tool → Final Answer
EXTENSION_QUESTION = "請根據 flight_delays.xlsx 找出延誤超過 30 分鐘的航班，並依航班作業規範說明需要通知哪些單位與應留存哪些紀錄。"
extension_result = execute_agent(EXTENSION_QUESTION)
show(extension_result)

{
  "question": "請根據 flight_delays.xlsx 找出延誤超過 30 分鐘的航班，並依航班作業規範說明需要通知哪些單位與應留存哪些紀錄。",
  "plan": {
    "tools": [
      "excel_python_tool"
    ],
    "resources": [
      "flight_operations_guide"
    ],
    "reason": "需要使用 Excel 工具來篩選出延誤超過 30 分鐘的航班，並根據航班作業規範說明需要通知的單位與應留存的紀錄，因此需要 flight_operations_guide 資源。"
  },
  "resources": {
    "flight_operations_guide": "航班作業規範（虛構教學資料）\n12. 航班延誤原因應使用核准分類，不得以未確認資訊對外說明。"
  },
  "generated_codes": {
    "excel_python_tool": {
      "code": "df = pd.read_excel(excel_path)\ndelayed_flights = df[df['delay_minutes'] > 30]\ndelayed_flights['notification'] = delayed_flights['department'].apply(lambda x: '通知航務部' if x == '航務部' else '通知地勤部')\ndelayed_flights['record'] = '延誤原因應使用核准分類，不得以未確認資訊對外說明'\nresult = delayed_flights[['flight', 'notification', 'record']]",
      "reason": "程式碼讀取 Excel 檔案，篩選出延誤超過 30 分鐘的航班，並根據航班作業規範，對不同部門的航班設定通知單位與紀錄說明。"
    }
  },
  "tool_results": {
    "excel_python_tool": "  flight notification                    record\n0  CI101    